# Laboratorium 02 — klasyfikacja na współśrodkowych okręgach z MLP

Materiał towarzyszący [`README.md`](README.md). Inne dane niż na wykładzie
(okręgi, nie półksiężyce), własna implementacja `nn.Module` i porównanie
architektur/aktywacji.




In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")


## 1. Dane: dwa współśrodkowe okręgi (gotowe)

Pierścień zewnętrzny (promień 1) to klasa 0, okrąg wewnętrzny (promień
0,5) to klasa 1, oba z szumem gaussowskim.


In [ ]:
def make_circles(n_samples, noise, factor, seed):
    g = torch.Generator().manual_seed(seed)
    n0 = n_samples // 2
    n1 = n_samples - n0

    theta0 = 2 * torch.pi * torch.rand(n0, generator=g)
    outer = torch.stack([torch.cos(theta0), torch.sin(theta0)], dim=1)

    theta1 = 2 * torch.pi * torch.rand(n1, generator=g)
    inner = factor * torch.stack([torch.cos(theta1), torch.sin(theta1)], dim=1)

    X = torch.cat([outer, inner], dim=0)
    y = torch.cat([torch.zeros(n0), torch.ones(n1)])
    X = X + noise * torch.randn(X.shape[0], 2, generator=g)

    perm = torch.randperm(X.shape[0], generator=g)
    return X[perm], y[perm]


X, y = make_circles(n_samples=320, noise=0.12, factor=0.5, seed=0)

n = X.shape[0]
perm = torch.randperm(n)
n_test = int(0.2 * n)
test_idx, train_idx = perm[:n_test], perm[n_test:]

X_train, y_train = X[train_idx].to(device), y[train_idx].to(device)
X_test, y_test = X[test_idx].to(device), y[test_idx].to(device)

print(f"N_train={len(train_idx)}, N_test={len(test_idx)}")

colors = ["tab:blue" if v == 0 else "tab:orange" for v in y]
plt.figure(figsize=(4.5, 4.5))
plt.scatter(X[:, 0], X[:, 1], c=colors, s=15, alpha=0.7)
plt.title("Dwa współśrodkowe okręgi")
plt.xlabel("x1")
plt.ylabel("x2")
plt.gca().set_aspect("equal")
plt.tight_layout()
plt.show()


## 2. Metryki — Twoje zadanie

Klasa 1 (okrąg wewnętrzny) to klasa pozytywna.


In [ ]:
def confusion_counts(preds, y_true):
    """Zwraca (tp, fp, tn, fn) dla predykcji 0/1 i etykiet 0/1."""
    # TODO: policz TP, FP, TN, FN porownaniami tensorowymi (==, &) i .sum().
    raise NotImplementedError("TODO: confusion_counts")


def metrics_from_preds(preds, y_true):
    """Zwraca dict {"accuracy", "precision", "recall"}."""
    # TODO: uzyj confusion_counts i wzorow:
    # accuracy = (tp+tn)/(tp+fp+tn+fn), precision = tp/(tp+fp), recall = tp/(tp+fn)
    # (uwaznaj na dzielenie przez zero, gdy mianownik to 0)
    raise NotImplementedError("TODO: metrics_from_preds")


def predict(model, X):
    """Zwraca predykcje 0/1 (prog 0.5 na sigmoidzie logitow modelu)."""
    # TODO
    raise NotImplementedError("TODO: predict")


def evaluate(model, X, y_true):
    """Zwraca metrics_from_preds(predict(model, X), y_true)."""
    # TODO
    raise NotImplementedError("TODO: evaluate")


## 3. Baseline (gotowe)

Pojedyncza warstwa liniowa — ten sam rodzaj modelu, co na wykładzie, na
innych danych.


In [ ]:
torch.manual_seed(0)
baseline_model = nn.Linear(2, 1).to(device)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.05)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(400):
    baseline_optimizer.zero_grad()
    logits = baseline_model(X_train).squeeze(-1)
    loss = loss_fn(logits, y_train)
    loss.backward()
    baseline_optimizer.step()

baseline_metrics = evaluate(baseline_model, X_test, y_test)
print(f"Baseline (liniowy) na zbiorze testowym: {baseline_metrics}")


## 4. Twój MLP — Twoje zadanie

Zaimplementuj klasę `MLP(nn.Module)` z jedną warstwą ukrytą o
konfigurowalnym rozmiarze i konfigurowalnej aktywacji.


In [ ]:
class MLP(nn.Module):
    """MLP: Linear(2, hidden_size) -> activation -> Linear(hidden_size, 1)."""

    def __init__(self, hidden_size, activation=nn.ReLU):
        super().__init__()
        # TODO: zdefiniuj warstwy jako atrybuty self (np. self.layer1 = ...).
        # `activation` to klasa (np. nn.ReLU) - utworz jej instancje: activation().
        raise NotImplementedError("TODO: MLP.__init__")

    def forward(self, x):
        # TODO: przepusc x przez warstwe 1, aktywacje, warstwe 2. Zwroc logity
        # (bez sigmoid - BCEWithLogitsLoss sama go liczy, patrz wyklad).
        raise NotImplementedError("TODO: MLP.forward")


## 5. Funkcja trenująca — Twoje zadanie

Uogólnij pętlę treningową z wykładu, żeby działała na dowolnym przekazanym
jej modelu `nn.Module`.


In [ ]:
def train_model(model, X_train, y_train, epochs=400, lr=0.05, seed=0):
    """Trenuje `model` (dowolny nn.Module) BCEWithLogitsLoss + Adam.

    Zwraca (model, loss_history).
    """
    # TODO: forward -> strata (BCEWithLogitsLoss) -> backward -> optimizer.step()
    # w petli po `epochs`, zbierajac strate kazdej epoki do listy. Patrz
    # analogiczna funkcja w lectures/02_mlp/examples.ipynb (train_classifier).
    raise NotImplementedError("TODO: train_model")


## 6. Eksperyment: domyślny MLP kontra baseline (gotowe, korzysta z Twojego kodu)


In [ ]:
default_mlp, default_loss_hist = train_model(
    MLP(hidden_size=16, activation=nn.ReLU), X_train, y_train
)
default_metrics = evaluate(default_mlp, X_test, y_test)

print(f"Baseline (liniowy):        {baseline_metrics}")
print(f"MLP (hidden=16, ReLU):     {default_metrics}")


## 7. Porównanie architektur/aktywacji — Twoje zadanie

Wybierz co najmniej trzy konfiguracje (różna liczba neuronów ukrytych
i/lub różna aktywacja), w tym co najmniej jedną, co do której podejrzewasz,
że będzie za słaba. Wytrenuj każdą i zbierz metryki testowe.


In [ ]:
# TODO: zdefiniuj liste >= 3 konfiguracji (np. slownikow z hidden_size i
# activation), wytrenuj kazda train_model(MLP(...), X_train, y_train) i
# zbierz metryki testowe (evaluate) w liscie `results`, wypisujac je po drodze.
raise NotImplementedError("TODO: porownanie konfiguracji")


## 8. Ewaluacja — Twoje zadanie

Zinterpretuj tabelę z sekcji 7 (komórka Markdown, 3–5 zdań): która
konfiguracja była najlepsza, czy któraś była wyraźnie za słaba, i dlaczego
— w kategoriach tego, co dana architektura może, a czego nie może
reprezentować.


_(Twoja odpowiedź tutaj.)_


## 9. Analiza błędnych klasyfikacji — Twoje zadanie

Granica decyzyjna domyślnego MLP z sekcji 6, z zaznaczonymi błędnie
sklasyfikowanymi punktami testowymi (kod gotowy).


In [ ]:
def plot_errors(model, X, y_true):
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = torch.meshgrid(
        torch.linspace(x_min, x_max, 200),
        torch.linspace(y_min, y_max, 200),
        indexing="xy",
    )
    grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1).to(X.device)
    with torch.no_grad():
        probs = torch.sigmoid(model(grid).squeeze(-1)).cpu().reshape(xx.shape)

    preds = predict(model, X)
    correct = preds == y_true

    plt.figure(figsize=(5.5, 5))
    plt.contourf(xx.cpu(), yy.cpu(), probs, levels=20, cmap="RdBu_r", alpha=0.6, vmin=0, vmax=1)
    plt.scatter(
        X[correct, 0].cpu(), X[correct, 1].cpu(),
        c=["tab:blue" if v == 0 else "tab:orange" for v in y_true[correct].cpu()],
        s=15, edgecolor="black", linewidth=0.3, label="poprawne",
    )
    plt.scatter(
        X[~correct, 0].cpu(), X[~correct, 1].cpu(),
        c="lime", marker="x", s=80, linewidth=2, label="błędne",
    )
    plt.gca().set_aspect("equal")
    plt.legend()
    plt.title("Błędne klasyfikacje na zbiorze testowym (MLP domyślny)")
    plt.tight_layout()
    plt.show()


plot_errors(default_mlp, X_test, y_test)


**Zadanie (3–5 zdań, komórka Markdown):** gdzie leżą błędnie sklasyfikowane
punkty względem granicy decyzyjnej? Czy wyglądają jak punkty "z natury
trudne" (blisko prawdziwej granicy, ofiary szumu), czy raczej jak dowód, że
model systematycznie czegoś nie rozumie? Uzasadnij, patrząc na wykres.

_(Twoja odpowiedź tutaj.)_


## 10. Wnioski — Twoje zadanie

Odpowiedz (markdown, uzasadnione, nie jednowersowe):

1. Czy wynik eksperymentu z sekcji 7 pasuje do tego, czego uczy dowód o
   złożeniu przekształceń afinicznych z wykładu? Czym różni się to, co
   pokazuje ten eksperyment, od tego, co pokazywał dowód?

   _(Twoja odpowiedź tutaj.)_

2. Gdybyś miał/miała jedno zdanie rady dla kogoś projektującego architekturę
   MLP do nowego problemu klasyfikacji, co by to było — na podstawie tego,
   co zaobserwowałeś/aś w tym laboratorium, nie z ogólników?

   _(Twoja odpowiedź tutaj.)_
